# Intro

## Motivation

In the previous modules, we built search engines and RAG pipelines. We tried different approaches: keyword search with minsearch, vector search, agents with function calling. But we never answered the obvious question of which one is actually better.

We could try a few queries by hand and see what looks good. That's fine for a quick sanity check, but it doesn't scale, and it doesn't give us a number to compare. We need a systematic way to tell whether one approach beats another.

That's what evaluation is for. And it's worth saying up front: of everything in this course, evaluation is the part that matters most. It's also the most tedious. But it's the only way to be sure your system works. And it's how you keep it working as you change prompts and swap models.

## The evaluation setup

### Search

For search evaluation, we need a dataset of questions where we know which document is the correct answer. We'll use an LLM to generate these questions from our FAQ data.

The approach works like this:

-   A = the original answer in the FAQ
-   Q* = a question generated from that answer by an LLM
-   We send Q* through our search and check if the original document appears in the results

### RAG

For RAG evaluation, we go one step further:

-   A = the original answer in the FAQ
-   Q* = a question generated from that answer by an LLM
-   A' = the answer produced by our RAG system when given Q*
-   We compare A' with A to see if the system produced the right answer

This is the A → Q* → A' pattern. We know the answer for each generated question because we created the question from that answer.

## Evaluation application and types

With evaluation, we can:

-   Compare different search methods (minsearch vs vector search vs hybrid)
-   Tune parameters (boost values, number of results, prompt templates)
-   Compare different LLMs (gpt-5.4-mini vs others)
-   Track improvements over time

There are two types of evaluation:

-   Offline evaluation: run the system on a test dataset and compute metrics
-   Online evaluation: collect feedback from real users in production

Offline evaluation is what we do before putting changes in front of users. It lets us compare search settings, prompts, or models on the same dataset. Online evaluation happens after deployment. It uses real traffic, feedback, logs, and dashboards to monitor quality.

In this module, **we focus on offline evaluation**. We'll generate a test dataset, run our search and RAG systems on it, and measure how well they perform.

Synthetic data is a good starting point when you don't have real user data. But generated questions can be too similar to the original FAQ text, which inflates the metrics. **As soon as you can**, start collecting real user queries and use them to validate your evaluation framework.

## What is covered?

We'll cover three levels of evaluation:

1.  Search evaluation: does the search return the right documents?
2.  RAG evaluation: does the LLM generate good answers?
3.  Agent evaluation: does the agent use tools efficiently?

Most of our time goes to search, and that's on purpose. Everything else depends on it: if retrieval brings back the wrong documents, no prompt or model can rescue the answer. So we test search on its own first, then evaluate the full pipeline on top of it.

- For search, we'll use two metrics:
    - Hit Rate
    - MRR (Mean Reciprocal Rank).
- For RAG quality, we'll use LLM-as-a-judge.
- For agents, we'll look at the final answer and the tool-call trajectory.

# Load dependencies

New [evaluation_util](../src/evaluation_utils.py) module is added for evaluation.

It contains helper functions we'll reuse in this module:
- `llm_structured`: calls the OpenAI API with structured output
- `llm_structured_retry`: retries structured-output calls when a
  request fails
- `calc_price`: calculates the price from token usage
- `calc_total_price`: calculates the total price from multiple usage
  objects
- `map_progress`: runs work in parallel and tracks progress. We'll use it
  in the next lesson.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys, json
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Literal
from openai import OpenAI
from pprint import pprint
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner
load_dotenv()

sys.path.append(Path().absolute().parent._str)
from src.data.ingest import load_faq_data
from src.data.index import build_index
from src.evaluation_utils import (
    calc_price, llm_structured, calc_total_price,
    map_progress, llm_structured_retry, calc_price,
    RAGWithUsage
)

In [ ]:
openai_client = OpenAI()

# Generating Ground Truth Data

To evaluate search, we need a dataset of queries where we know which
document is the correct answer. This is called ground truth (or gold
standard) data.

For each query in our ground truth dataset, we know which document in
the knowledge base is relevant. When we run a search, we check whether
the results include the correct document.

There are several ways to get ground truth data:

- Human annotators look at documents and write queries (best quality, expensive)
- Collect real user queries and label them (requires a running system)
- Generate synthetic data with an LLM (what we'll do)

We don't have a production system yet, so we'll use an LLM to generate
questions. For each FAQ document, we ask the LLM to create 5 questions
that this document would answer. Then we know that for each generated
question, the source document is the correct answer.

## Loading the documents

We'll generate questions only for the LLM Zoomcamp FAQ. The full FAQ dataset contains documents from multiple courses. Generating five questions for every document would take longer and cost more.

We'll use these documents from now on so let's name them as `documents`

In [ ]:
documents = load_faq_data()
documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)
documents = documents_llm


Each document already has an `id` field:



In [ ]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

- The ID becomes the label in our ground truth dataset.
    - We generate questions from a document, so we know that this document holds the answer.
    - Later, search evaluation checks whether search brings back the document with this ID.
- This is why every record needs a stable ID. If you can't uniquely identify a document, you can't tell whether search retrieved the right one.

## Generating questions with structured output

- We use an LLM to generate questions for each document.

- This is the first time we're using **structured output** in the course.
    - With structured output, we ask the LLM to return data in a specific format instead of free-form text.
    - For example, instead of getting a paragraph that contains questions, we can ask for a Python object with a `questions` field.
- This is useful when code will process the output. The model returns the same structure every time. We can access the generated questions directly instead of parsing text manually.
- We want the output as a list of strings, so we define that structure with a **Pydantic model**:

In [ ]:
class Questions(BaseModel):
    questions: list[str]


The instructions for the LLM:



In [ ]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record.
The record should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions on the internet. Not too formal, not too short, not too long.
""".strip()

We ask the LLM to use different wording from the original document. This makes the evaluation more realistic - real users won't phrase their questions the same way as the FAQ.

- Call the LLM for one document.
- Prepare the document as JSON
- Create the messages

In [ ]:
user_prompt = json.dumps(doc)

messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

- Until now we called `responses.create` and read `response.output_text`.
- For **structured output** we switch to `responses.parse` and pass `text_format=Questions`, which tells the API to return our class instead of free text.
- The parsed object is available in `response.output_parsed`
- You should see 5 questions that relate to the first FAQ document.

In [ ]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)
result = response.output_parsed

In [ ]:
pprint(result)
print("==================")
pprint(result.questions)

- retrying the same exaple using the [llm_structured](../src/evaluation_utils.py#L31-L44) function
- The response also contains token usage
- As in the agents module, we calculate the price from `response.usage`.

In [ ]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

In [ ]:
pprint(result.questions)
print("================")
print(usage.input_tokens, usage.output_tokens)
print("================")
cost = calc_price(usage)
print(cost)

## Convert questions into ground truth records

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

print(records)

Each record has two fields:

- `question`: the question generated by the LLM
- `document`: the ID of the FAQ document that should answer the question

The `document` field connects the generated question to the document that contains the answer. Later, when we evaluate search, we'll ask the search engine the generated question. Then we'll check if it retrieves the document with this ID.

We now know how to generate and store questions for one document.

# Generating Ground Truth for All Documents

- In the previous lesson, we generated questions for one document and converted them into ground truth records.
- We want to do the same thing for every document in the FAQ dataset. For each document, we generate questions and save them as ground truth records.

The processing function takes one document and turns it into ground
truth records.

For each document, we:

- convert the document to JSON so we can send it to the LLM
- ask the LLM to return a `Questions` object
- create one ground truth record for each generated question

Each record contains the generated question and the ID of the document
that should answer the question.

When we send many requests, one of them might fail. We don't want the entire batch to fail because of one temporary error.

`llm_structured` makes one structured-output call. `llm_structured_retry` wraps the same call in a retry loop. If one request fails because of a temporary API or network issue, it waits briefly and tries again.

In [ ]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [ ]:
ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

This works, but it runs one LLM call after another. Running it for all documents this way would take too long.

## Parallel processing

Running the calls one after another wastes most of the time waiting on the network. Each request just sits there until OpenAI responds, so we can fire several at once and wait on them together. We process the documents in parallel and track progress while the requests run.

**One caution**: don't open too many connections at once, or you'll hit the provider's rate limits. Five or six workers is a safe default here.

This submits one job per document, updates the progress bar when a job finishes, and collects the results. If you want a more detailed explanation of `ThreadPoolExecutor` and futures, ask ChatGPT to walk through this helper line by line.

Then replace the loop with the parallel version.

`generate_ground_truth` returns two things for each document: the generated records and the token usage.

Split those into separate lists.

With 5 questions per document, you should get roughly 5x the number of documents.

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

In [ ]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

print(f"ground_truth count: {len(ground_truth)}")
print(f"total cost: {calc_total_price(usages)}")

Create a dataframe so we can look at the records as a table and save them as a CSV file.

Because we generated the questions from specific documents, we know which document is correct for each question. We now have the ground truth we need for evaluation.

Save it for later use.

In [ ]:
df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv("ground_truth-new.csv", index=False)

# Search Evaluation

- Now that we have ground truth data, we can evaluate how well our search retrieves the correct documents.
- For each question in our ground truth dataset, we run search. Then we check whether the results include the correct document.

## Setting up search

- For search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.
- Load the documents and build a minsearch index.
- Wrap the search call in a function called `text_search`. The name is deliberate. Later we'll write `vector_search` or a hybrid version and run the exact same evaluation on it.
- Everything downstream only needs a function that takes a query and returns results, so we can swap one for another.
- That mirrors how RAG works: the retrieval step doesn't care which search function sits behind it.

In [ ]:
df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

documents = load_faq_data()
documents = [
    doc for doc in documents
    if doc["course"] == "llm-zoomcamp"
]

index = build_index(documents)

def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

## Collecting relevance data

- Start with one ground truth record.
- Run search for this question
- First, compare the retrieved document IDs with the correct document ID. Then turn this comparison into a relevance list.
- In this lesson, relevance means whether a retrieved document is the correct document for this question.
- This gives a list of `0` and `1` values. `1` means the retrieved document has the same ID as the correct document.

In [ ]:
q = ground_truth[0]
doc_id = q["document"]
results = text_search(query=q["question"])

for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

print(f"relevance: {relevance}")

- Put this logic into a function.

In [ ]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [ ]:
compute_relevance_text(q)

- The correct document was the first search result.
- Here are two more examples from the generated ground truth data.

In [ ]:
q = ground_truth[11]
print(q["question"])
print(compute_relevance_text(q))
q = ground_truth[50]
print(q["question"])
print(compute_relevance_text(q))

- Now do the same thing for all ground truth questions.
- Call it for the first 15 ground truth questions.

In [ ]:
def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)
pprint(relevance_total_text)

- Each entry in `relevance_total_text` is a relevance list. This is enough to check that the function works before we run it for the full dataset.
- Next, make the relevance functions generic. We start with text search, but later we may want to evaluate vector search, hybrid search, or another retrieval method.
- The relevance logic is the same. Only the search function changes.

In [ ]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
pprint(relevance_total)

- Now run it for all ground truth questions.
- Now we can represent search results as relevance lists. Next, we'll turn these lists into metrics: Hit Rate and MRR.

In [ ]:
relevance_total = compute_relevance_total(ground_truth, text_search)

# Search Evaluation Metrics

## Hit Rate

- **Hit Rate** (also called Recall@k) measures the fraction of queries where the correct document appears anywhere in the results:
- In our setup, each query has one correct document, so Hit Rate and **Recall@k** mean the same thing.

In [ ]:
example = relevance_total[:30]

In [ ]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [ ]:
print(hit_rate(example))

## Mean Reciprocal Rank (MRR)

- Hit Rate tells us if we found the right document, but not where it was.
- MRR also considers the **position**.
- For each query, the score is based on the rank of the first correct document:
    - position 1: score is 1.0
    - position 2: score is 0.5
    - position 3: score is 0.333
    - not found: score is 0

- MRR is the average of these scores across all queries. It rewards systems that put the correct document near the top.
- Hit Rate is the upper bound for MRR. In practice, MRR is usually smaller because some correct documents are found below the first position.

In [ ]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [ ]:
print(mrr(example))

## Putting it together


We can evaluate any search function:



In [ ]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [ ]:
evaluate(
    ground_truth,
    text_search
)


Search metrics tell us whether retrieval works. Next, we'll use these
metrics to tune the search parameters.



## Interpreting the metrics

A few things to keep in mind when reading these numbers:

- Our ground truth assumes only one relevant document per query. In practice, other retrieved documents might also be relevant.
    - A 50% hit rate does not mean that half the results are useless. It means the document we generated the question from did not appear in the top results for half the queries. Other relevant documents may still be there.

- With synthetic data, the generated questions can be too close to the original FAQ text.
    - This inflates hit rate and MRR.
    - If you see numbers above 95%, treat them with caution and check whether the questions are realistic enough.

- Good thresholds depend on your use case. A 50% hit rate is acceptable for some applications, while others need 90% or higher.
    - The right number depends on how much the downstream LLM can compensate for imperfect retrieval.
    - It also depends on user tolerance for wrong answers.

- Look at the system holistically.
    - A high MRR means the relevant document is near the top, which helps the LLM focus on the right context.
    - A low MRR with a high hit rate means the document is there, but buried under less relevant results.

# Search Parameter Tuning

- In the previous section, we defined Hit Rate, MRR, and the `evaluate` function. Now we can use them to tune search parameters.
- Instead of guessing which settings are better, we measure them on the ground truth dataset.
- So far we've boosted `question` to 3.0. The idea was that a query should match the FAQ question. That kind of match should count for more than matching the answer text.
    - It sounds reasonable. But it's a guess, and now we can check it against data instead of trusting it.
- This is the main benefit of offline evaluation. We change one parameter, run the same questions again, and see whether the metric moves.
- The dataset stays fixed, so the comparison is fair.

## Trying different boosts

- Start with a search function where the question boost is configurable.
- Evaluate several boost values.

In [ ]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

- Increasing the question boost makes the metrics worse, not better. The best value here is `1.0`, no boost at all. That's already the opposite of what the intuition predicted.
- But this is only one parameter. We can also tune `answer` and `section` together with `question`.
- Then do a small grid search.

In [ ]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [ ]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })


In [ ]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

- The best combination weights `answer` twice as heavily as `question`, with almost no weight on `section`.
- So the data says the opposite of where we started. The answer text matters more for retrieval than the question text. The intuition was wrong, and we'd never have known without measuring it. This is exactly why we evaluate instead of guess.
- So we can use the smaller and easier-to-read values: `question=0.5`, `answer=2.0`, and `section=0.1`.
    - This gives the same relative weights but the numbers are not unnecessarily large.

In [ ]:
def text_search(query):
    boost_dict = {
        "question": 0.5,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

## Tuning Workflow

- Search parameters can look arbitrary. This includes field boosts, number of results, filters, and other settings. Evaluation gives us a way to compare settings with evidence.
- Grid search is fine when there are only a few settings. For a larger parameter space, use a smarter search strategy.
    - You can sample random combinations, use Bayesian optimization, or keep a validation split so you don't overfit the evaluation set.
- For text search on our dataset, grid search takes about one second per combination. That makes it practical to try many options. When each evaluation takes minutes instead of seconds, grid search becomes too expensive. In those cases, use **Bayesian optimization** with a library like **hyperopt**. It explores the parameter space more efficiently by focusing on combinations that are likely to improve the metric.

## Top-K tradeoffs

- We return 5 results from search. Increasing top-K to 10 would improve hit rate because there are more chances to find the correct document.
- But more results means more context sent to the LLM. That costs more and makes it harder for the model to identify what is relevant.
- Five results is a reasonable default for short FAQ-style documents.

# RAG and Agent Evaluation

- So far, we evaluated retrieval. We checked whether search returns the document that should answer the question.
- That is only the first step. A complete application still needs to produce a final answer.
    - For RAG, this means checking the generated answer.
    - For agents, it also means looking at the tool calls the model made before producing the answer.
- RAG evaluation checks the whole flow together:
    - search
    - prompt
    - LLM

- If the final answer is bad, the problem can come from any of these steps.
    - The search might retrieve the wrong document,
    - the prompt might omit important context,
    - or the LLM might ignore the context.

- In this part, we'll evaluate:
    - RAG answers with an LLM judge
    - Agent answers and tool-call trajectories
        - We won't go deep into agent evaluation frameworks here.
        - We'll use the agent from module 01, save the final answer, and also save the tool calls.
        - Then we can look at whether the answer is good and whether the trajectory looks reasonable.

## LLM as a judge

- For RAG and agent evaluation, we compare the generated answer with the original answer.
- The generated answer won't use the same words as the original. It's a generative model, so the phrasing will be different even when the meaning is the same.
- This is why we use another LLM to do the comparison.
    - We show the judge the question, the original answer, and the generated answer.
    - Then we ask it to decide if they are semantically equivalent.
- This approach is called LLM-as-a-judge. The evaluating LLM is the judge. It classifies each answer as good or bad and explains its reasoning.
- Asking the judge to explain why it made a decision generally produces better classifications than asking for just the verdict.

# Generating RAG Answers

- For each generated question, we run RAG and save the answer produced by the LLM.
- Later, we'll compare this answer with the original FAQ answer.
- This is the A->Q->A' setup:
    - A = original answer in the FAQ
    - Q = generated question from this answer
    - A' = answer produced by our RAG system
- If A' is close to A, the RAG system is doing a good job.
- This is still **offline** evaluation.
    - We can compare A and A' because our questions came from FAQ records.
    - For each question, we know which original answer it came from.

## Loading the data

- Load the ground truth questions.
- Load the FAQ documents and the search index.
- Create a lookup table for the original FAQ documents.
    - We'll use this lookup table to find the original answer for each ground truth question.

In [ ]:
df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

documents = load_faq_data()
documents = [
    doc for doc in documents
    if doc["course"] == "llm-zoomcamp"
]

index = build_index(documents)
doc_idx = {doc["id"]:doc for doc in documents}

## Running RAG

- `RAGWithUsage` subclasses `RAGBase`
- It stores token usage after each LLM call. Then we can calculate the total cost later.
- It also uses the search boosts we selected in the search tuning lesson:
    - `question=0.5`,
    - `answer=2.0`,
    - `section=0.1`.
- For each question, `RAGBase` searches the FAQ, builds a prompt with the retrieved context, and asks the LLM to answer.
- We save the answer for later judgement.

In [ ]:
with open("templates/instructions.txt", "r") as file:
    instructions = file.read().strip()
with open("templates/prompt_template.txt", "r") as file:
    prompt_template = file.read().strip()

In [ ]:
assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    instructions = instructions,
    prompt_template = prompt_template,
    search_boost_dict = {"question": 1.0, "answer": 2.0, "section": 0.1},
)

rec = ground_truth[0]
question = rec["question"]
answer_llm = assistant.rag(question)

doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]


rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}
pprint(rag_result)

print(f"cost: {assistant.total_cost()}")

## Processing all questions

In [ ]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [ ]:
answer_record = generate_rag_answer(ground_truth[0])
pprint(answer_record)

- Before running the full batch, reset the usage we collected while testing:
- This calls the LLM once per ground truth question, so it can take some time. Let's process the questions in parallel and track progress.
- `generate_rag_answer` returns one answer record for each question. Collect the answer records
- Calculate the total cost at the end and save the answers

In [ ]:
assistant.reset_usage()

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

In [ ]:
assistant.total_cost() # 0.5313945000000004

In [ ]:
df_answers = pd.DataFrame(results)
df_answers.to_csv("rag-answers-new.csv", index=False)

# LLM as a Judge

- For offline evaluation, we have three things:
- the original FAQ answer
- the question generated from that answer
- the answer generated by our RAG pipeline

- An LLM judge is another LLM call that compares these three pieces. We ask it whether the RAG answer recovers the same information as the original answer. It can also explain why an answer is bad. For example:
    - the retrieved document might be wrong
    - the answer might miss the key point
    - the model might say that it doesn't know

- This approach is useful when exact text matching is too strict. The RAG answer doesn't need to copy the FAQ answer word for word. It needs to answer the question with the same key information.

- This evaluates the full RAG flow in one pass:
    - search: did we retrieve context that contains the answer?
    - prompt: did we give the model enough context to answer?
    - LLM: did the model produce a useful answer from that context?

- If the judge marks an answer as bad, we still need to look at the example. The judge tells us where to investigate. It doesn't replace reading the failing cases.

## Loading the RAG answers

In [ ]:
df_answers = pd.read_csv("rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

- Each row has the generated question, the original FAQ answer, and the answer produced by the RAG pipeline.
- This is offline evaluation. We can do it because our test dataset came from FAQ records. We know the original answer for every generated question.
- In production, we usually don't have that original answer for real user questions. There we can still use an LLM judge. The prompt has to judge only the question and the generated answer. In this lesson, we use the stronger offline setup.

## A->Q->A' evaluation

- We'll compare the RAG answer with the original answer from the FAQ. This checks if the RAG pipeline is producing answers that match the ground truth.
- First, define the output format.
- The judge returns two fields. The `score` gives us a metric we can aggregate. The `reasoning` explains the score, which helps when we look at bad examples.
- First, write the judge instructions. This tells the judge what to compare and how to assign the score.
- Then define the prompt template. This is the data we pass to the judge for each answer.
- try it for a single example and calculate the cost

In [ ]:
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [ ]:
rec = answers[0]
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)
pprint(eval_result)
print(calc_price(usage))

- Now put the same logic into a function
- Test it on the same record.

In [ ]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [ ]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

pprint(eval_result)

## Running the judge for all sample records

In [ ]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

- Use the same parallel processing helper.
- Split the results.
- Create a dataframe.
- Calculate the total cost.
- Check the results.
- Look at the "bad" cases to understand what went wrong.

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

In [ ]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

df_eval = pd.DataFrame(evaluations)

print(f"total cost: {calc_total_price(usages)}")

good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

The following rows are often the most useful part of the evaluation.
- They can show that search retrieved the wrong document.
- They can also show that the answer is too generic.
- Sometimes the RAG pipeline says that it doesn't know even though the FAQ had the answer.

In [ ]:
df_eval[df_eval["score"] == "bad"].head()

## Evaluating the judge

- The judge can be wrong. It may rate an answer as good even though search failed to retrieve the right document. In that case the judge is too lenient. Make the instructions stricter and re-run the evaluation.
- To evaluate the judge, you need to look at the results yourself.
    - Sample some good and bad cases, read the judge reasoning, and check whether you agree with the verdict.
    - You cannot use another judge to evaluate the judge. This is manual work, but it is necessary.
- **A practical approach** is to build a simple application with Streamlit. Show each question, the original answer, the generated answer, and the judge verdict side by side. Then mark each verdict as correct or incorrect and use that feedback to adjust the judge instructions. This is a lot of trial and error, but it makes the evaluation framework more reliable. (a great TODO)

## Saving the results

In [ ]:
df_eval.to_csv("rag-evaluations-new.csv", index=False)

# Agent Evaluation

- For RAG, we used the A->Q->A' setup:
    - A = original answer in the FAQ
    - Q = generated question from this answer
    - A' = answer produced by our RAG system
- For agents, we use the same setup. A' comes from an agent instead of a fixed RAG pipeline.
- We also save the trajectory. Here, the **trajectory** means only the tool calls the agent made before producing the final answer.

## Loading the data

In [ ]:
df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

documents = load_faq_data()
documents = [
    doc for doc in documents
    if doc["course"] == "llm-zoomcamp"
]

index = build_index(documents)
doc_idx = {doc["id"]:doc for doc in documents}

## Running the agent

- Reuse the ToyAIKit agent from module 01. It handles the agent loop and stores the full message history.
- Define the search tool.
- Create the runner.
- The result contains:
    - `last_message`: the final response
    - `all_messages`: the full message history
    - `cost`: the cost of all LLM calls in this run
- Run it for one ground truth question.

In [ ]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

rec = ground_truth[0]
result = runner.loop(prompt=rec["question"])

- Look at the full message history.
- For this lesson, the trajectory is only the tool calls. We don't need to send the full message history to the judge.
- Extract the function name and arguments.
- Get the original answer.
- Save the A->Q->A' record and the trajectory.
- The `answer_agent` field is what we evaluate with the LLM judge. The `tool_calls` field lets the judge see how the agent got there.

In [ ]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [ ]:
print(f"all messages: {result.all_messages}")
print("="*40)
tool_calls = extract_tool_calls(result.all_messages)
print(f"tool_calls: {tool_calls}")
print("="*40)
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

pprint(agent_result)

In [ ]:
tc = extract_tool_calls(result.all_messages)

In [ ]:
type(tc)

## Processing multiple questions

- Create a function that processes one ground truth record.
- Run it for a small sample in parallel.
- Turn it into a dataframe.
- Calculate the total cost.
- Save the results.

In [ ]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:24], generate_agent_answer)

In [ ]:
df_agent = pd.DataFrame(agent_answers)
print(f"total cost: {df_agent["cost"].sum()}")
df_agent.to_csv("agent-answers.csv", index=False)

- Then load it.
- Our judge can look at both:
    - whether `answer_agent` matches `answer_orig`
    - whether the tool calls look reasonable for the question
- This lets us evaluate the final answer and the agent behavior in one place.

In [ ]:
df_agent = pd.read_csv("agent-answers.csv")
df_agent.tool_calls = df_agent.tool_calls.map(eval)
agent_answers = df_agent.to_dict(orient="records")

## Judging answers and trajectories

- A good trajectory is not just "many tool calls". A good trajectory uses the available tools in a way that helps answer the question.
- For our search agent, a good trajectory has these properties:
    - The search query is relevant to the user question
    - The query includes the important keywords from the question
    - The agent avoids duplicate searches with the same arguments
    - If it searches more than once, the next query is a useful refinement
    - It usually uses 1 search call
    - 2-3 calls can be okay for harder questions
    - More than 3 search calls needs a clear reason
    - The tool calls support the final answer
    - The agent does not stop too early or keep searching without a reason

- Define a judge output type with two scores.
- Then the judge instructions.
- Then, define the judge function.
- Test it on one agent result.

In [ ]:
class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [ ]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [ ]:
def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [ ]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])
print(agent_eval)

- When the answer is bad, the trajectory score tells us whether the problem started with tool use.
    - If the answer is bad but the trajectory is good, the model may have used the retrieved context poorly.
    - If both are bad, the agent likely searched for the wrong thing. It may also have stopped too early.

## Running the agent judge

- Run the judge for all agent answers
- Use the same parallel helper
- Split the results
- Create a dataframe
- Calculate the judge cost from the token usage
- Check the answer scores
- Check the trajectory scores
- Save the judge result

In [ ]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [ ]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

In [ ]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)


df_agent_eval = pd.DataFrame(agent_evaluations)
print(f"total cost: {calc_total_price(usages)}")
print(f"answer score distribution:\n {df_agent_eval["answer_score"].value_counts()}")
print(f"trajetory score distribution:\n {df_agent_eval["trajectory_score"].value_counts()}")

In [ ]:
df_agent_eval.to_csv("agent-evaluations.csv", index=False)

# Next Steps

## From synthetic data to real data

- The evaluation workflow in practice:
   1. Start with synthetic data.
      - Use an LLM to generate questions from your FAQ or documentation. This gives you a baseline without needing real users.
   2. Tune the data generation.
      - If the metrics look suspiciously good, the synthetic questions may be too close to the source text.
      - Adjust the generation prompt to produce more realistic questions.
   3. Deploy and collect real data.
      - Once the system is in production, start collecting actual user queries and feedback.
   4. Label real data.
      - Have humans label whether the retrieved documents and generated answers are correct.
      - This produces the most reliable ground truth.
   5. Tune synthetic generation to match real data.
      - Use the patterns from real queries to improve your synthetic data generator.
      - The closer your synthetic data is to real data, the more useful the metrics become.
- Nothing beats manual evaluation.
   - Try the system yourself, think about edge cases, and collect examples of where it fails.
   - This is especially important in the early stages when you don't have automated evaluation set up yet.

## Evaluation frameworks

- For production systems, consider using evaluation frameworks that make it easier to manage test datasets, run evaluations, and track results:
  - Ragas: a framework for evaluating RAG systems with metrics like faithfulness, answer relevance, and context precision
  - DeepEval: provides built-in metrics for RAG evaluation including hallucination detection and answer relevance
  - TruLens: instruments your LLM app and tracks quality metrics
- These frameworks implement many of the concepts we covered here and add visualizations and experiment tracking.